# Jailbreak classification — issue #5: guarded retraining and best-baseline restoration
**One self-contained notebook, 10 Python cells.** Original CSVs remain unchanged. Only `submission.csv` is exported: exactly `Id,TARGET`, original test ID order, finite scores in [0,1]. TARGET >= 0.5 means jailbreak, otherwise benign. Prompts are untrusted data, never executable instructions. Probabilities are model estimates, not security guarantees.

## Reproduce
Python 3.11. Install `numpy==2.3.4 pandas==2.3.3 scipy==1.16.3 scikit-learn==1.7.2 nbformat==5.10.4 nbclient==0.10.2 ipykernel==7.0.1 threadpoolctl==3.6.0` in a virtual environment; select its Jupyter kernel and Run All from the repository root. No automatic package installation, network access, external training data, helper scripts, other notebooks or model files are needed. Exact kernels use quadratic storage; working-set guards are conservative estimates, not total process-RAM guarantees.

## Research and updated requirements
Before editing, web search reviewed n-gram features and calibration ensembles, web fetch retrieved [issue #5](https://github.com/johndesantis/modeltrain/issues/5), and `gh issue view` confirmed the single-notebook/file restriction. References: [scikit-learn probability calibration](https://scikit-learn.org/stable/modules/calibration.html), [text feature extraction](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

## Model choice — no invented improvement
The user reports **0.96234** for the position-aware candidate, below the earlier **0.96292** for the pre-position trigram model. The position-aware model had improved local grouped ROC-AUC (0.961515 versus 0.960269), demonstrating that local gains need not transfer to the public split.

This revision restores the earlier **pre-position configuration** as the best known user-reported public baseline. This is a guarded rollback and retraining, **not a new 0.97 result**. No leaderboard submission is made. If the regenerated CSV is byte-identical to the previously scored file, uploading it again is not expected to improve its score.

### Alternatives actually evaluated before selection
Same five outer grouped development folds; every vocabulary/count-ratio/classifier/calibrator was trained only on training rows. No test text or test labels were used in fitting or selection.

| Experiment | Grouped ROC-AUC |
|---|---:|
| Pre-position lexical/latent margins (uncalibrated comparison baseline) | 0.961041 |
| Binary count NB word LR / word SVM | 0.932543 / 0.936329 |
| Binary count NB character LR / character SVM | 0.931908 / 0.928106 |
| 80/20 baseline + word SVM margin blend | 0.960985 |
| 80/20 baseline + character SVM margin blend | 0.959346 |
| Group-calibrated 3-member bagging (probabilities) | 0.957406 |
| Group-calibrated 5-member bagging (probabilities) | 0.959565 |

Count ratios used binary document presence and training labels only; character counts used 3–5 grams. Sigmoid ensemble=True candidates used inner group-disjoint folds with independently fitted TF-IDF/latent models. They were rejected because they did not outperform the baseline. These are measured development experiments, not rerun by this notebook on every execution. Selected-model metrics below are recomputed every run. Earlier iterations also tried BM25, IDF/TF variations, feature selection, position views and latent ranks.

The same development folds and public feedback have been reused across revisions, so selection optimism is unavoidable. Neither these results nor the seed-2026 sensitivity split constitutes an untouched holdout. The bootstrap interval is conditional on fitted OOF predictions and excludes training/selection uncertainty. **The public 0.97 target remains unmet/unverified.**

## Restored model
Word 1–3-gram TF-IDF and character-boundary 2–6-gram TF-IDF, min_df=1, sublinear TF, 180k features per view and equal L2-normalized view weighting. Two C10 polynomial SVMs use lexical cosine and a training-only 256-dimensional latent eigensystem, blended 80/20 before sigmoid calibration. Three group-disjoint inner folds refit every vocabulary, IDF, latent basis and classifier; ensemble=False then refits the classifier on all available training rows. Outer evaluation uses five grouped folds, final scoring uses all labeled data.

## Safeguards and regression tests
Strict CSV structure/schema/label validation, canonical Unicode IDs, normalized conflicting duplicates/overlap checks and invalid Unicode/control rejection. Files are parsed from verified SHA-256 byte snapshots; exports recheck original hashes. Label-free character-hash cosine groups at 0.80 stay together in outer and inner folds. Mixed-label groups and train/test near-matches are audit-only, never relabeled or copied. Lexical grouping cannot guarantee semantic/source independence.

New checks reject non-string CSV inputs and surrogate IDs, validate both columns of each predicted probability distribution (shape, finite values, bounds and sum-to-one), and reject output hardlinks to any input. New fault-injection tests cover **fsync failures, atomic-replacement failures and corrupt CSV read-back**, in addition to disk write failures and changing source files. Each failure must preserve the previous successful submission and remove temporary files. Tests verify full-data classifier refitting and deployment of only the intended word/character views.

The exporter uses an exclusively created temporary stream, flush/fsync, strict ID/score read-back and hash validation before atomic replacement. Existing snapshot, allocation, inference immutability, unknown-word-ID, order/batch invariance and NA/NULL-ID roundtrip tests remain. No extra output files survive. Passing the tests demonstrates execution and checked software behavior, **not 100% predictive accuracy, universal input coverage or 0.97 leaderboard performance**. All documentation, reports, checks and outputs live in this notebook.

## Verified outcome
Fresh-kernel execution completed without cell errors: **102 checks passed**, 1,000 valid submission rows, unchanged original input hashes. Selected-model grouped development ROC-AUC is **0.960269**. The regenerated `submission.csv` is **byte-identical** to commit `b2d8e72`, the earlier file the user reported at **0.96292**. This reproduces that baseline; it is not a new public result and resubmitting identical bytes is not expected to improve the score. Runtime was approximately 6.4 minutes. **No evaluated alternative achieved 0.97.**


In [1]:
from pathlib import Path
import csv, hashlib, json, platform, re, time, unicodedata, warnings
import numpy as np
import pandas as pd
import scipy, sklearn
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, brier_score_loss
from sklearn.neighbors import NearestNeighbors
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
warnings.filterwarnings('error', category=ConvergenceWarning)
SEED = 42
FOLDS = 5
NEAR_DUPLICATE_COSINE = 0.80
ROOT = Path.cwd()
START = time.monotonic()
# Avoid oversubscribed BLAS on CPU environments.
thread_limit = threadpool_limits(limits=2)
print('Python', platform.python_version(), '| sklearn', sklearn.__version__)
from io import StringIO
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import Normalizer
from sklearn.svm import SVC
INPUT_HASHES = {f: hashlib.sha256((ROOT/f).read_bytes()).hexdigest()
                for f in ['train.csv','test.csv','sample_submission.csv']}

import tempfile, os
from sklearn.metrics import average_precision_score


Python 3.11.2 | sklearn 1.7.2


## 1. Strict CSV validation (IDs are never predictive features)

In [2]:
def require(condition, message):
    if not condition:
        raise ValueError(message)

def normalize(text):
    text = unicodedata.normalize('NFKC', text).casefold()
    text = ''.join(c for c in text if unicodedata.category(c) != 'Cf')
    return ' '.join(text.split())

def validate_frame(frame, columns):
    require(list(frame.columns) == columns, f'Expected columns {columns}')
    require(len(frame) > 0, 'Empty dataset')
    require(not frame.isna().any().any(), 'Missing values')
    require(all(frame[col].map(lambda value: isinstance(value, str)).all() for col in columns),
            'CSV input fields must be strings')
    require(frame['Id'].str.strip().ne('').all(), 'Blank IDs')
    require(frame['Id'].is_unique, 'Duplicate IDs')
    require(not frame['Id'].map(lambda s: any(0xD800 <= ord(ch) <= 0xDFFF for ch in s)).any(),
            'Unpaired Unicode surrogate in ID')
    canonical_ids = frame['Id'].map(lambda s: unicodedata.normalize('NFKC', s))
    require(canonical_ids.is_unique, 'Unicode-equivalent duplicate IDs')
    require(not frame['Id'].map(lambda s: any(unicodedata.category(ch) == 'Cf' for ch in s)).any(),
            'Invisible formatting in IDs')
    require(frame['Id'].eq(frame['Id'].str.strip()).all(), 'Whitespace in IDs')
    require(not frame['Id'].str.contains(r'[\x00-\x1f\x7f]', regex=True).any(), 'Control character in ID')
    if 'text' in columns:
        require(frame.text.map(normalize).ne('').all(), 'Empty normalized prompts')
        require(not frame.text.str.contains('\x00', regex=False).any(), 'NUL in prompt')
        require(not frame.text.str.contains(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', regex=True).any(),
                'Nontext control character in prompt')
        require(not frame.text.map(lambda s: any(0xD800 <= ord(ch) <= 0xDFFF for ch in s)).any(),
                'Unpaired Unicode surrogate in prompt')
    if 'label' in columns:
        require(set(frame.label) == {'benign', 'jailbreak'}, 'Expected both binary labels')
    return frame

def parse_csv(stream, columns):
    reader = csv.reader(stream, strict=True)
    try:
        require(next(reader, None) == columns, 'Invalid CSV header')
        rows = list(reader)
    except csv.Error as exc:
        raise ValueError('Malformed CSV quoting') from exc
    require(all(len(row) == len(columns) for row in rows), 'Malformed CSV row width')
    return validate_frame(pd.DataFrame(rows, columns=columns, dtype=str), columns)

def parse_verified_snapshot(data, expected_sha256, columns):
    require(hashlib.sha256(data).hexdigest() == expected_sha256, 'CSV snapshot hash mismatch')
    # Parse precisely the bytes that were verified, not a second read of a mutable path.
    return parse_csv(StringIO(data.decode('utf-8'), newline=''), columns)

def load_csv(name, columns):
    return parse_verified_snapshot((ROOT/name).read_bytes(), INPUT_HASHES[name], columns)

def validate_relations(train, test, sample):
    require(set(train.Id.map(lambda s: unicodedata.normalize('NFKC', s))).isdisjoint(
                test.Id.map(lambda s: unicodedata.normalize('NFKC', s))), 'Train/test canonical ID overlap')
    require(sample.Id.tolist() == test.Id.tolist(), 'Sample/test ID order mismatch')
    require(set(train.text.map(normalize)).isdisjoint(test.text.map(normalize)),
            'Normalized train/test text overlap')
    require(train.assign(key=train.text.map(normalize)).groupby('key').label.nunique().max() == 1,
            'Conflicting labels for identical normalized prompts')
    sample_scores = pd.to_numeric(sample.TARGET, errors='coerce').to_numpy()
    require(np.isfinite(sample_scores).all() and ((sample_scores >= 0) & (sample_scores <= 1)).all(),
            'Invalid sample targets')

train = load_csv('train.csv', ['Id', 'text', 'label'])
test = load_csv('test.csv', ['Id', 'text'])
sample = load_csv('sample_submission.csv', ['Id', 'TARGET'])
validate_relations(train, test, sample)
train['normalized'] = train.text.map(normalize)
test['normalized'] = test.text.map(normalize)
require(set(train.normalized).isdisjoint(test.normalized), 'Normalized train/test text overlap')
conflicts = train.groupby('normalized').label.nunique()
require(conflicts.max() == 1, 'Conflicting labels for identical normalized prompts')
duplicate_rows = int(train.normalized.duplicated().sum())
train = train.drop_duplicates('normalized').reset_index(drop=True)
texts = train.normalized.to_numpy()
y = train.label.map({'benign': 0, 'jailbreak': 1}).to_numpy()
print('Training:', len(train), '| test:', len(test), '| removed duplicates:', duplicate_rows)
print(train.label.value_counts().to_dict())

Training: 4000 | test: 1000 | removed duplicates: 0
{'benign': 2401, 'jailbreak': 1599}


## 2. Conservative near-duplicate grouping
Unsupervised hashing is used **only** for split grouping, never as predictive features. Connected components keep transitive template families in one fold. This is a lexical safeguard, not a semantic/source-independence guarantee.

In [3]:
from sklearn.feature_extraction.text import HashingVectorizer
# No learned vocabulary or IDF, no test prompts and no labels used for grouping.
group_vectorizer = HashingVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                    n_features=2**18, alternate_sign=False, norm='l2')
group_matrix = group_vectorizer.transform(texts)
neighbor_index = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=2).fit(group_matrix)
graph = neighbor_index.radius_neighbors_graph(group_matrix,
    radius=1 - NEAR_DUPLICATE_COSINE + 1e-12, mode='connectivity')
_, groups = connected_components(graph, directed=False)
splits = list(StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
              .split(texts, y, groups))
for tr, va in splits:
    require(set(groups[tr]).isdisjoint(groups[va]), 'Group leakage')
    require(len(np.unique(y[tr])) == len(np.unique(y[va])) == 2, 'Single-class fold')
print('Groups:', len(np.unique(groups)), '| largest:', int(np.bincount(groups).max()))
print('Validation fold sizes:', [len(va) for _, va in splits])
# Audit only; test similarities never determine model weights or predictions.
nearest_distance, _ = neighbor_index.kneighbors(
    group_vectorizer.transform(test.normalized), n_neighbors=1)
near_test_count = int((nearest_distance[:, 0] <= 1 - NEAR_DUPLICATE_COSINE).sum())
print('Test prompts near a training prompt (audit only):', near_test_count)

# Flag mixed-label near-duplicate families for human review; never relabel or drop them.
mixed_groups = int((pd.DataFrame({'group':groups, 'label':y}).groupby('group').label.nunique() > 1).sum())
print('Mixed-label lexical groups (retained, kept within folds):', mixed_groups)
print('Prompt character lengths, train/test medians:',
      float(train.text.str.len().median()), float(test.text.str.len().median()))


Groups: 3855 | largest: 85
Validation fold sizes: [729, 803, 793, 791, 884]


Test prompts near a training prompt (audit only): 43
Mixed-label lexical groups (retained, kept within folds): 0
Prompt character lengths, train/test medians: 616.0 606.5


## 3. Restored lexical + latent baseline and complete probability validation
The pre-position model uses only word/character TF-IDF, with no truncated positional blocks. It combines lexical and training-only latent polynomial SVM margins 80/20. Grouped calibration refits features/bases/classifiers inside every inner training fold, followed by a full-data classifier refit. The complete probability matrix (including the benign column) is checked for shape, finiteness, bounds and normalization. IDs are never predictive features.

In [4]:
def vectorizer(view):
    common = dict(sublinear_tf=True, min_df=1, max_df=1.0, dtype=np.float64)
    if view == 'word':
        return TfidfVectorizer(ngram_range=(1, 3), token_pattern=r'(?u)\b\w+\b',
                               max_features=180000, **common)
    return TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6),
                           max_features=180000, **common)

def polynomial_cosine(X, Y):
    return (1.0 + (X @ Y.T).toarray()) ** 2

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.preprocessing import normalize as l2_normalize
from scipy.linalg import eigh

MAX_KERNEL_WORKING_BYTES = 2 * 1024**3

def check_kernel_budget(training_rows, prediction_rows=0, maximum=MAX_KERNEL_WORKING_BYTES):
    # Conservative dense-matrix working-set estimate, not a total process-RAM guarantee.
    require(training_rows >= 2 and prediction_rows >= 0, 'Invalid kernel dimensions')
    estimate = 8 * (4 * training_rows**2 + 3 * training_rows * prediction_rows)
    require(estimate <= maximum, f'Estimated dense-kernel working set {estimate:,} exceeds budget {maximum:,}')
    return estimate

class LexicalLatentSVM(ClassifierMixin, BaseEstimator):
    """Two training-only polynomial SVMs; fixed 80/20 margin blend.

    Eigenvectors of the training cosine Gram matrix implement exact truncated
    LSA. Validation/test projections use the training basis, never a joint SVD.
    """
    def __init__(self, C=10.0, latent_components=256, latent_weight=0.2, random_state=42):
        self.C = C
        self.latent_components = latent_components
        self.latent_weight = latent_weight
        self.random_state = random_state

    def fit(self, X, y):
        require(X.shape[0] == len(y), 'Feature/label length mismatch')
        require(np.isfinite(X.data).all(), 'Nonfinite training features')
        require(0 <= self.latent_weight <= 1, 'Invalid latent blend weight')
        require(isinstance(self.latent_components, int) and self.latent_components >= 1,
                'Invalid latent rank')
        check_kernel_budget(X.shape[0])
        self.training_features_ = X.copy().tocsr()
        self.n_features_in_ = X.shape[1]
        self.classes_ = np.unique(y)
        require(self.classes_.tolist() == [0, 1], 'Expected both model classes')
        cosine = (X @ X.T).toarray()
        rank = min(self.latent_components, X.shape[0] - 1)
        values, vectors = eigh(cosine, subset_by_index=[X.shape[0]-rank, X.shape[0]-1],
                               check_finite=True)
        values = np.maximum(values, 1e-10)
        self.projection_ = vectors / np.sqrt(values)
        self.latent_training_ = l2_normalize(vectors * np.sqrt(values))
        self.lexical_ = SVC(C=self.C, kernel='precomputed', random_state=self.random_state)
        self.latent_ = SVC(C=self.C, kernel='precomputed', random_state=self.random_state)
        self.lexical_.fit((1.0 + cosine)**2, y)
        self.latent_.fit((1.0 + self.latent_training_ @ self.latent_training_.T)**2, y)
        return self

    def decision_function(self, X):
        check_is_fitted(self, ['lexical_', 'latent_', 'projection_'])
        require(X.shape[1] == self.n_features_in_, 'Inference feature dimension mismatch')
        require(np.isfinite(X.data).all(), 'Nonfinite inference features')
        check_kernel_budget(self.training_features_.shape[0], X.shape[0])
        cosine = (X @ self.training_features_.T).toarray()
        latent = l2_normalize(cosine @ self.projection_)
        lexical_margin = self.lexical_.decision_function((1.0 + cosine)**2)
        latent_margin = self.latent_.decision_function((1.0 + latent @ self.latent_training_.T)**2)
        return (1.0-self.latent_weight)*lexical_margin + self.latent_weight*latent_margin

    def predict(self, X):
        return self.classes_[(self.decision_function(X) >= 0).astype(int)]

def make_pipeline():
    return Pipeline([
        ('features', FeatureUnion([('word',vectorizer('word')), ('char',vectorizer('char'))])),
        ('normalize', Normalizer()),
        ('svm', LexicalLatentSVM(C=10, latent_components=256, latent_weight=.2, random_state=SEED))])

def validate_probability_matrix(probabilities, row_count):
    probabilities = np.asarray(probabilities)
    require(probabilities.shape == (row_count, 2), 'Invalid binary probability shape')
    require(np.isfinite(probabilities).all(), 'Nonfinite binary probability')
    require(((probabilities >= 0) & (probabilities <= 1)).all(), 'Binary probability outside [0,1]')
    require(np.allclose(probabilities.sum(axis=1), 1, rtol=0, atol=1e-10),
            'Binary probabilities do not sum to one')
    return probabilities[:, 1]

def fit_model(train_text, labels, prediction_text, train_groups):
    inner = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
                 .split(train_text, labels, train_groups))
    for tr, va in inner:
        require(set(train_groups[tr]).isdisjoint(train_groups[va]), 'Inner group leakage')
        require(len(np.unique(labels[tr])) == len(np.unique(labels[va])) == 2,
                'Inner fold lacks a class')
    model = CalibratedClassifierCV(make_pipeline(), method='sigmoid', cv=inner,
                                  ensemble=False, n_jobs=1)
    model.fit(train_text, labels)
    require(model.classes_.tolist() == [0, 1], 'Incorrect class direction')
    return validate_probability_matrix(model.predict_proba(prediction_text), len(prediction_text)), model


## 4. Five-fold out-of-fold evaluation (no test feedback)

In [5]:
oof_score = np.full(len(train), np.nan)
fold_assignment = np.full(len(train), -1)
fold_metrics = []
for fold, (tr, va) in enumerate(splits):
    oof_score[va], _ = fit_model(texts[tr], y[tr], texts[va], groups[tr])
    fold_assignment[va] = fold
    auc = roc_auc_score(y[va], oof_score[va])
    fold_metrics.append(float(auc))
    print(f'Fold {fold+1}: ROC-AUC={auc:.6f}', flush=True)
require(np.isfinite(oof_score).all(), 'Incomplete OOF predictions')
oof = oof_score[:, None]
metrics = {'lexical_latent_svm': float(roc_auc_score(y, oof_score))}
print('Grouped OOF ROC-AUC:', metrics)
print('Accuracy:', accuracy_score(y, oof_score >= .5),
      '| F1:', f1_score(y, oof_score >= .5), '| Brier:', brier_score_loss(y, oof_score))
print('Public ROC-AUC: unknown; 0.97 target remains unverified')
print('Average precision:', average_precision_score(y, oof_score))

# Predeclared seed-shift sensitivity check, not an independent untouched holdout.
# Do not change model settings based on this result; all rows were used during development.
sensitivity_split = next(StratifiedGroupKFold(5, shuffle=True, random_state=2026).split(texts,y,groups))
sensitivity_tr, sensitivity_va = sensitivity_split
require(set(groups[sensitivity_tr]).isdisjoint(groups[sensitivity_va]), 'Sensitivity group leakage')
sensitivity_score, _ = fit_model(texts[sensitivity_tr], y[sensitivity_tr],
                               texts[sensitivity_va], groups[sensitivity_tr])
sensitivity_auc = float(roc_auc_score(y[sensitivity_va], sensitivity_score))
print('Seed-2026 grouped split sensitivity ROC-AUC:', sensitivity_auc,
      '| validation rows:', len(sensitivity_va), flush=True)


Fold 1: ROC-AUC=0.962173


Fold 2: ROC-AUC=0.970808


Fold 3: ROC-AUC=0.962311


Fold 4: ROC-AUC=0.960497


Fold 5: ROC-AUC=0.955808


Grouped OOF ROC-AUC: {'lexical_latent_svm': 0.9602685351814272}
Accuracy: 0.89025 | F1: 0.8569566634082764 | Brier: 0.07854543485036831
Public ROC-AUC: unknown; 0.97 target remains unverified
Average precision: 0.9452450204571674


Seed-2026 grouped split sensitivity ROC-AUC: 0.9710649750096118 | validation rows: 739


## 5. Group bootstrap uncertainty
Resampling whole groups accounts for lexical dependence, conditional on these fitted OOF predictions. It does not include model-selection/training uncertainty and is not a public-score interval.

In [6]:
rng = np.random.default_rng(SEED)
group_rows = [np.flatnonzero(groups == g) for g in np.unique(groups)]
bootstrap_auc = []
for _ in range(500):
    idx = np.concatenate([group_rows[g] for g in rng.integers(0, len(group_rows), len(group_rows))])
    if len(np.unique(y[idx])) == 2:
        bootstrap_auc.append(roc_auc_score(y[idx], oof_score[idx]))
auc_interval = np.quantile(bootstrap_auc, [.025, .975]).tolist()
print('Group-bootstrap 95% interval:', auc_interval)

Group-bootstrap 95% interval: [0.9484298569564138, 0.9697414816724085]


## 6. Refit on all labeled rows and score untouched test prompts

In [7]:
scores, final_model = fit_model(texts, y, test.normalized.to_numpy(), groups)
require(np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all(), 'Invalid probabilities')
print('Test score range:', float(scores.min()), float(scores.max()))
print('Threshold counts:', pd.Series(np.where(scores >= .5, 'jailbreak', 'benign')).value_counts().to_dict())

Test score range: 6.407572380247664e-05 0.9999933748836289
Threshold counts: {'benign': 605, 'jailbreak': 395}


## 7. Fitted word-ID inspection
The unigram entries of the final word vocabulary supply stable IDs within this fitted model. Repeated words share IDs; unknown words map to -1. IDs are feature indices, not prompt IDs, and do not encode risk on their own.

In [8]:
final_pipeline = final_model.calibrated_classifiers_[0].estimator
word_vect = dict(final_pipeline.named_steps['features'].transformer_list)['word']
analyzer = word_vect.build_tokenizer()
def word_ids(prompt):
    return [(word, word_vect.vocabulary_.get(word, -1))
            for word in analyzer(normalize(prompt))]
print('Vocabulary entries (word 1–3 grams):', len(word_vect.vocabulary_))
print(word_ids(train.text.iloc[0])[:25])

Vocabulary entries (word 1–3 grams): 180000
[('you', np.int64(178024)), ('are', np.int64(12090)), ('an', np.int64(7692)), ('ai', np.int64(6329)), ('developed', np.int64(64878)), ('to', np.int64(166860)), ('provide', np.int64(144583)), ('useful', np.int64(172065)), ('information', np.int64(122645)), ('to', np.int64(166860)), ('users', np.int64(172185)), ('i', np.int64(119007)), ('m', np.int64(129098)), ('conducting', np.int64(42086)), ('a', np.int64(1204)), ('study', np.int64(156078)), ('about', np.int64(4608)), ('data', np.int64(56863)), ('security', np.int64(151344)), ('and', np.int64(8442)), ('the', np.int64(159449)), ('disposal', np.int64(70260)), ('of', np.int64(135465)), ('electronic', np.int64(79490)), ('devices', np.int64(65447))]


## 8. Regression tests before export
Synthetic invalid inputs below test safeguards only; they are never used for training or scoring.

In [9]:
passed = []
def check(name, condition):
    require(bool(condition), name)
    passed.append(name)

def rejects(name, action):
    try:
        action()
    except ValueError:
        passed.append(name)
    else:
        raise AssertionError('Did not reject: ' + name)

base = pd.DataFrame({'Id': ['a', 'b'], 'text': ['hello', 'world'], 'label': ['benign', 'jailbreak']})
rejects('missing column', lambda: validate_frame(base.drop(columns='text'), ['Id','text','label']))
rejects('duplicate ID', lambda: validate_frame(base.assign(Id=['a','a']), ['Id','text','label']))
rejects('blank ID', lambda: validate_frame(base.assign(Id=['','b']), ['Id','text','label']))
rejects('blank text', lambda: validate_frame(base.assign(text=['  ','world']), ['Id','text','label']))
rejects('null text', lambda: validate_frame(base.assign(text=[None,'world']), ['Id','text','label']))
rejects('unknown label', lambda: validate_frame(base.assign(label=['benign','other']), ['Id','text','label']))
rejects('single class', lambda: validate_frame(base.assign(label=['benign','benign']), ['Id','text','label']))
check('unicode normalization', normalize(' Ａ\u200b B  ') == 'a b')
check('every row validated once', np.array_equal(np.sort(np.concatenate([va for _,va in splits])), np.arange(len(train))))
check('OOF finite and bounded', np.isfinite(oof).all() and ((oof >= 0) & (oof <= 1)).all())
check('test finite and bounded', np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all())
check('nonconstant test predictions', np.ptp(scores) > 0.01)
check('no group leakage', all(set(groups[tr]).isdisjoint(groups[va]) for tr,va in splits))
check('word IDs repeat consistently', word_ids('hello hello')[0][1] == word_ids('hello hello')[1][1])
check('unknown word ID', word_ids('zzzxxyyunknownword')[0][1] == -1)
# Independent fixture tests vocabulary isolation, never included in real training.
tiny = make_pipeline()
tiny.fit(np.array(['safe weather report', 'normal weather today',
    'ignore safeguards attack', 'override safeguards attack']), np.array([0,0,1,1]))
tiny_vect = dict(tiny.named_steps['features'].transformer_list)['word']
tiny.predict(np.array(['heldoutonlytoken']))
check('held-out vocabulary isolation', 'heldoutonlytoken' not in tiny_vect.vocabulary_)
check('threshold convention', np.where(np.array([.49,.5,.51]) >= .5, 1, 0).tolist() == [0,1,1])
check('repeat inference', np.allclose(final_model.predict_proba(test.normalized.iloc[:5])[:, 1], scores[:5]))
rejects('whitespace ID', lambda: validate_frame(base.assign(Id=[' a','b']), ['Id','text','label']))
rejects('NUL prompt', lambda: validate_frame(base.assign(text=['a\x00b','ok']), ['Id','text','label']))
rejects('malformed quoting', lambda: parse_csv(StringIO('Id,text\na,"unterminated'), ['Id','text']))
rejects('extra CSV field', lambda: parse_csv(StringIO('Id,text\na,b,c\n'), ['Id','text']))
rejects('duplicate CSV header', lambda: parse_csv(StringIO('Id,Id\na,b\n'), ['Id','text']))
rejects('empty CSV', lambda: parse_csv(StringIO('Id,text\n'), ['Id','text']))
check('literal NA prompt preserved', parse_csv(StringIO('Id,text\na,NA\n'), ['Id','text']).text.iloc[0] == 'NA')
valid_test = pd.DataFrame({'Id':['c'], 'text':['different prompt']})
valid_sample = pd.DataFrame({'Id':['c'], 'TARGET':['0.5']})
rejects('overlapping IDs', lambda: validate_relations(base, valid_test.assign(Id=['a']), valid_sample.assign(Id=['a'])))
rejects('normalized text overlap', lambda: validate_relations(base, valid_test.assign(text=[' HELLO ']), valid_sample))
rejects('conflicting duplicate labels', lambda: validate_relations(base.assign(text=['Hello',' hello ']), valid_test, valid_sample))
rejects('sample ID mismatch', lambda: validate_relations(base, valid_test, valid_sample.assign(Id=['wrong'])))
rejects('sample nonfinite score', lambda: validate_relations(base, valid_test, valid_sample.assign(TARGET=['nan'])))
rejects('sample score out of bounds', lambda: validate_relations(base, valid_test, valid_sample.assign(TARGET=['1.1'])))
check('normalization idempotent', all(normalize(normalize(t)) == normalize(t) for t in train.text.iloc[:100]))
check('inner calibration group isolation', all(set(groups[a]).isdisjoint(groups[b]) for a,b in
    StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(texts,y,groups)))
check('inputs unchanged', all(hashlib.sha256((ROOT/f).read_bytes()).hexdigest() == h for f,h in INPUT_HASHES.items()))
print(f'PASS: {len(passed)} checks before export')
def validate_submission(frame, expected_ids):
    require(list(frame.columns) == ['Id', 'TARGET'], 'Invalid submission schema')
    require(len(frame) == len(expected_ids), 'Submission row count mismatch')
    require(frame.Id.tolist() == list(expected_ids), 'Submission ID order mismatch')
    require(frame.Id.is_unique, 'Duplicate submission IDs')
    require(pd.api.types.is_numeric_dtype(frame.TARGET) and not pd.api.types.is_bool_dtype(frame.TARGET),
            'Submission scores must be numeric, not boolean')
    values = frame.TARGET.to_numpy(dtype=float)
    require(np.isfinite(values).all(), 'Nonfinite submission scores')
    require(((values >= 0) & (values <= 1)).all(), 'Submission scores out of bounds')

fixture = pd.DataFrame({'Id':['x','y'], 'TARGET':[.2,.8]})
rejects('export reordered IDs', lambda: validate_submission(fixture.iloc[::-1], ['x','y']))
rejects('export missing row', lambda: validate_submission(fixture.iloc[:1], ['x','y']))
rejects('export extra column', lambda: validate_submission(fixture.assign(label='benign'), ['x','y']))
rejects('export nonnumeric score', lambda: validate_submission(fixture.assign(TARGET=['.2','bad']), ['x','y']))
rejects('export infinite score', lambda: validate_submission(fixture.assign(TARGET=[np.inf,.8]), ['x','y']))
rejects('export missing score', lambda: validate_submission(fixture.assign(TARGET=[np.nan,.8]), ['x','y']))
rejects('export negative score', lambda: validate_submission(fixture.assign(TARGET=[-.01,.8]), ['x','y']))
rejects('export score above one', lambda: validate_submission(fixture.assign(TARGET=[.2,1.01]), ['x','y']))
rejects('Unicode-equivalent duplicate IDs', lambda: validate_frame(base.assign(Id=['Ａ','A']), ['Id','text','label']))
rejects('invisible format ID', lambda: validate_frame(base.assign(Id=['a\u200bb','c']), ['Id','text','label']))
rejects('unpaired surrogate text', lambda: validate_frame(base.assign(text=['bad\ud800','ok']), ['Id','text','label']))
check('test row permutation invariance', np.allclose(
    final_model.predict_proba(test.normalized.iloc[:5].iloc[::-1])[:,1], scores[:5][::-1], atol=1e-12))
print(f'PASS: {len(passed)} checks before export including adversarial submission fixtures')

latent_estimator = final_pipeline.named_steps['svm']
check('latent basis finite', np.isfinite(latent_estimator.projection_).all())
check('latent rank', latent_estimator.projection_.shape == (len(train), 256))
check('latent training rows only', latent_estimator.training_features_.shape[0] == len(train))
# A row's projection does not depend on other rows being scored.
check('single versus batch scoring', np.allclose(
    final_model.predict_proba(test.normalized.iloc[:1])[:,1], scores[:1], rtol=0, atol=1e-12))
print(f'PASS: {len(passed)} checks before CSV roundtrip')

def export_submission(frame, expected_ids, output, input_hashes):
    """Validate before writing; failures leave an existing submission untouched."""
    validate_submission(frame, expected_ids)
    output = Path(output)
    require(not output.is_symlink(), 'Refusing symlinked output')
    require(output.resolve() not in [Path(f).resolve() for f in input_hashes], 'Output aliases input')
    require(not output.exists() or all(not os.path.samefile(output, Path(f)) for f in input_hashes),
            'Output hard-links an input')
    def verify_sources():
        require(all(hashlib.sha256(Path(f).read_bytes()).hexdigest() == h
                    for f,h in input_hashes.items()), 'Input contents changed during run')
    verify_sources()
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='',
                dir=output.parent, prefix='.submission-', suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            frame.to_csv(stream, index=False, float_format='%.12g')
            stream.flush()
            os.fsync(stream.fileno())
        restored = pd.read_csv(temporary, dtype={'Id':str}, keep_default_na=False)
        validate_submission(restored, expected_ids)
        require(np.allclose(restored.TARGET, frame.TARGET, rtol=0, atol=1e-11), 'CSV score roundtrip failed')
        verify_sources()
        temporary.replace(output)
        return restored
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

rejects('boolean TARGET rejected', lambda: validate_submission(fixture.assign(TARGET=[True,False]), ['x','y']))
rejects('nontext prompt control rejected', lambda: validate_frame(base.assign(text=['a\x07b','ok']), ['Id','text','label']))
# Isolated export fixtures are deleted immediately; no additional project files remain.
from unittest.mock import patch
with tempfile.TemporaryDirectory() as directory:
    directory = Path(directory)
    source = directory/'source.csv'
    source.write_text('unchanged source')
    hashes = {source:hashlib.sha256(source.read_bytes()).hexdigest()}
    dest = directory/'submission.csv'
    sentinel = b'previous successful submission\n'
    dest.write_bytes(sentinel)
    rejects('invalid export rejected', lambda: export_submission(fixture.assign(TARGET=[-1,.5]), ['x','y'], dest, hashes))
    check('invalid export preserves existing file', dest.read_bytes() == sentinel)
    rejects('source overwrite rejected', lambda: export_submission(fixture, ['x','y'], source, hashes))
    link = directory/'linked.csv'
    link.symlink_to(source)
    rejects('symlink export rejected', lambda: export_submission(fixture, ['x','y'], link, hashes))
    source.write_text('changed source')
    rejects('changed source aborts export', lambda: export_submission(fixture, ['x','y'], dest, hashes))
    check('source failure preserves existing file', dest.read_bytes() == sentinel)
    source.write_text('unchanged source')
    try:
        with patch.object(pd.DataFrame, 'to_csv', side_effect=OSError('simulated disk error')):
            export_submission(fixture, ['x','y'], dest, hashes)
    except OSError:
        passed.append('disk write failure propagated')
    else:
        raise AssertionError('Simulated disk failure was not propagated')
    check('write failure preserves existing file', dest.read_bytes() == sentinel)
    check('write failure removes temporary files', not list(directory.glob('.submission-*.tmp')))
    # Identifiers that pandas normally interprets as missing must round-trip literally.
    reserved = fixture.assign(Id=['NA','NULL'])
    restored = export_submission(reserved, ['NA','NULL'], dest, hashes)
    check('NA-like IDs preserved by real exporter', restored.Id.tolist() == ['NA','NULL'])
    check('successful export removes temporary files', not list(directory.glob('.submission-*.tmp')))

word_idf_before = word_vect.idf_.copy()
projection_before = latent_estimator.projection_.copy()
_ = final_model.predict_proba(np.array(['zzzxxyyunseenword', 'safe weather request']))
check('inference leaves IDF unchanged', np.array_equal(word_vect.idf_, word_idf_before))
check('inference leaves latent basis unchanged', np.array_equal(latent_estimator.projection_, projection_before))
print(f'PASS: {len(passed)} checks before production export')

# Snapshot integrity is tested with in-memory bytes, not by changing real inputs.
snapshot = b'Id,text\na,hello\n'
snapshot_hash = hashlib.sha256(snapshot).hexdigest()
check('verified snapshot accepted', parse_verified_snapshot(snapshot, snapshot_hash, ['Id','text']).Id.tolist() == ['a'])
rejects('changed snapshot rejected', lambda: parse_verified_snapshot(snapshot+b' ', snapshot_hash, ['Id','text']))
bad_utf8 = b'Id,text\na,\xff\n'
rejects('invalid UTF8 rejected', lambda: parse_verified_snapshot(bad_utf8, hashlib.sha256(bad_utf8).hexdigest(), ['Id','text']))
check('kernel budget accepts supplied data', check_kernel_budget(len(train), len(test)) < MAX_KERNEL_WORKING_BYTES)
rejects('excessive kernel allocation rejected', lambda: check_kernel_budget(100000))
rejects('invalid kernel dimensions rejected', lambda: check_kernel_budget(-1))
rejects('invalid latent weight rejected', lambda: LexicalLatentSVM(latent_weight=2).fit(csr_matrix(np.eye(4)), np.array([0,0,1,1])))
rejects('invalid latent rank rejected', lambda: LexicalLatentSVM(latent_components=0).fit(csr_matrix(np.eye(4)), np.array([0,0,1,1])))
print(f'PASS: {len(passed)} checks before export, including snapshot and allocation guards')



rejects('numeric input ID rejected', lambda: validate_frame(base.assign(Id=[1,2]), ['Id','text','label']))
rejects('surrogate input ID rejected', lambda: validate_frame(base.assign(Id=['bad\ud800','ok']), ['Id','text','label']))
rejects('probability row count rejected', lambda: validate_probability_matrix(np.array([[.2,.8]]), 2))
rejects('probability single column rejected', lambda: validate_probability_matrix(np.array([[.8]]), 1))
rejects('negative complementary probability rejected', lambda: validate_probability_matrix(np.array([[-.1,.8]]), 1))
rejects('nonfinite complementary probability rejected', lambda: validate_probability_matrix(np.array([[np.nan,.8]]), 1))
rejects('non-normalized probability pair rejected', lambda: validate_probability_matrix(np.array([[.1,.8]]), 1))
check('valid binary distribution accepted', validate_probability_matrix(np.array([[.2,.8]]), 1).tolist() == [.8])
with tempfile.TemporaryDirectory() as directory:
    directory = Path(directory)
    source = directory/'source.csv'
    source.write_bytes(b'original input')
    source_bytes = source.read_bytes()
    hashes = {source:hashlib.sha256(source_bytes).hexdigest()}
    alias = directory/'hardlinked.csv'
    os.link(source, alias)
    rejects('hard-linked input export rejected', lambda: export_submission(fixture, ['x','y'], alias, hashes))
    check('hard-linked source preserved', source.read_bytes() == source_bytes)
    destination = directory/'submission.csv'
    sentinel = b'previous successful submission\n'
    destination.write_bytes(sentinel)
    for stage, target, attribute in [
        ('fsync', os, 'fsync'), ('replace', Path, 'replace')]:
        try:
            with patch.object(target, attribute, side_effect=OSError('simulated '+stage+' error')):
                export_submission(fixture, ['x','y'], destination, hashes)
        except OSError:
            passed.append(stage+' failure propagated')
        else:
            raise AssertionError(stage+' failure was not propagated')
        check(stage+' failure preserves prior submission', destination.read_bytes() == sentinel)
        check(stage+' failure cleans temporary file', not list(directory.glob('.submission-*.tmp')))
    with patch.object(pd, 'read_csv', return_value=fixture.assign(Id=['bad','ids'])):
        rejects('corrupt readback aborts export', lambda: export_submission(fixture, ['x','y'], destination, hashes))
    check('corrupt readback preserves prior submission', destination.read_bytes() == sentinel)
    check('corrupt readback cleans temporary file', not list(directory.glob('.submission-*.tmp')))
print(f'PASS: {len(passed)} checks before production export')

check('full-data classifier refit', latent_estimator.training_features_.shape[0] == len(train))
check('only lexical feature views deployed', [name for name,_ in final_pipeline.named_steps['features'].transformer_list] == ['word','char'])
check('one full-data calibration classifier', len(final_model.calibrated_classifiers_) == 1)
final_distribution = final_model.predict_proba(test.normalized.iloc[:5])
check('complete prediction distribution valid', np.allclose(
    validate_probability_matrix(final_distribution, 5), scores[:5], rtol=0, atol=1e-12))
print(f'PASS: {len(passed)} checks before final production export')


PASS: 34 checks before export
PASS: 46 checks before export including adversarial submission fixtures


PASS: 50 checks before CSV roundtrip
PASS: 65 checks before production export
PASS: 73 checks before export, including snapshot and allocation guards
PASS: 92 checks before production export


PASS: 96 checks before final production export


## 9. Atomic submission export and inline audit report
The only exported file is `submission.csv`, exactly `Id,TARGET`, preserving test ID spelling/order. All checks, input hashes, versions, metrics, and limitations are printed below and saved as notebook outputs (no report sidecar).

In [10]:
submission = pd.DataFrame({'Id': test.Id, 'TARGET': scores})
roundtrip = export_submission(submission, test.Id.tolist(), ROOT/'submission.csv',
    {ROOT/f:h for f,h in INPUT_HASHES.items()})
check('CSV schema', roundtrip.columns.tolist() == ['Id','TARGET'])
check('CSV ID order', roundtrip.Id.tolist() == sample.Id.tolist())
check('CSV row count', len(roundtrip) == len(test))
check('CSV probabilities', np.allclose(roundtrip.TARGET, scores, rtol=0, atol=1e-11))
check('CSV probability bounds', roundtrip.TARGET.between(0,1).all())
check('input hashes verified after export', all(
    hashlib.sha256((ROOT/f).read_bytes()).hexdigest() == h for f,h in INPUT_HASHES.items()))
report = {
    'model': '80/20 lexical/latent polynomial SVM, full-data refit with grouped sigmoid calibration',
    'selection': 'Restore best user-reported public baseline; reject weaker development alternatives',
    'best_prior_user_reported_public_score': .96292,
    'kernel_working_set_limit_bytes': MAX_KERNEL_WORKING_BYTES,
    'latent_components': 256, 'min_df': 1, 'word_ngram_range': [1,3],
    'sensitivity_split_seed': 2026, 'sensitivity_split_roc_auc': sensitivity_auc,
    'seed': SEED, 'folds': FOLDS, 'training_rows': len(train), 'test_rows': len(test),
    'removed_exact_duplicates': duplicate_rows,
    'class_counts': train.label.value_counts().to_dict(), 'near_test_prompts_audit': near_test_count, 'mixed_label_lexical_groups': mixed_groups,
    'near_duplicate_cosine': NEAR_DUPLICATE_COSINE, 'groups': len(group_rows),
    'development_roc_auc': metrics, 'fold_roc_auc': fold_metrics,
    'group_bootstrap_95_interval': auc_interval,
    'accuracy_at_0.5': float(accuracy_score(y, oof_score >= .5)),
    'f1_at_0.5': float(f1_score(y, oof_score >= .5)),
    'brier': float(brier_score_loss(y, oof_score)),
    'average_precision': float(average_precision_score(y, oof_score)),
    'tests_passed': passed,
    'input_sha256': {f: hashlib.sha256((ROOT/f).read_bytes()).hexdigest()
                     for f in ['train.csv','test.csv','sample_submission.csv']},
    'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__},
    'submission_sha256': hashlib.sha256((ROOT/'submission.csv').read_bytes()).hexdigest(),
    'public_leaderboard_score': None, 'user_reported_previous_public_score': 0.96234,
    'limitations': ['Lexical grouping cannot guarantee semantic or source independence.',
                    'Likelihoods are not independently calibrated.',
                    'Model family and hyperparameters compared on reused development folds.',
                    'Public leaderboard target 0.97 is unverified.'],
}
print(json.dumps(report, indent=2))
print(f'PASS: {len(passed)} checks; exported {len(submission)} rows to submission.csv')
print(f'Runtime: {time.monotonic()-START:.1f} seconds')

{
  "model": "80/20 lexical/latent polynomial SVM, full-data refit with grouped sigmoid calibration",
  "selection": "Restore best user-reported public baseline; reject weaker development alternatives",
  "best_prior_user_reported_public_score": 0.96292,
  "kernel_working_set_limit_bytes": 2147483648,
  "latent_components": 256,
  "min_df": 1,
  "word_ngram_range": [
    1,
    3
  ],
  "sensitivity_split_seed": 2026,
  "sensitivity_split_roc_auc": 0.9710649750096118,
  "seed": 42,
  "folds": 5,
  "training_rows": 4000,
  "test_rows": 1000,
  "removed_exact_duplicates": 0,
  "class_counts": {
    "benign": 2401,
    "jailbreak": 1599
  },
  "near_test_prompts_audit": 43,
  "mixed_label_lexical_groups": 0,
  "near_duplicate_cosine": 0.8,
  "groups": 3855,
  "development_roc_auc": {
    "lexical_latent_svm": 0.9602685351814272
  },
  "fold_roc_auc": [
    0.9621729282199485,
    0.9708077987055101,
    0.9623108849912975,
    0.9604967080590049,
    0.9558082951394783
  ],
  "group_boots